[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter2/generative-llms.ipynb)

# Chapter 2: Generative LLMs for RAG

In a RAG pipeline, the generative LLM is the final step — it takes retrieved context and the user's query, then produces a natural language answer. The quality of this generation step depends heavily on the prompt template and the model's ability to stay grounded in the provided context.

This notebook demonstrates using Anthropic's Claude to generate responses in a RAG pipeline with prompt templates.

**What you'll learn:**
- Structure a RAG prompt template with query and context placeholders
- Use the Anthropic API to generate grounded responses
- Test how the same context answers different queries

**Prerequisites:** `pip install anthropic python-dotenv` and set `ANTHROPIC_API_KEY` in a `.env` file.

## Setup

We load API credentials and define a prompt template with placeholders for the query and retrieved context.

In [1]:
# Save ANTHROPIC_API_KEY in a .env file

import dotenv
dotenv.load_dotenv()

import anthropic

In [2]:

query = "If I don't care about budget or time, what's the most accurate way to judge the RAG system?"

context = """
The next step is to build the critics for evaluating the responses from RAG. A common practice is called **LLM-as-a-judge**. 
Once again, we leverage the instruction following capability of LLMs to configure them into evaluators that judge different aspects of the responses. 

There are usually two common aspects: 
**relevance** -- does the response answer the query? and 
**faithfulness** -- is the response well supported by the retrieved documents? 

LLM-as-a-judge has several drawbacks. 
First, it maybe slow and expensive because it uses expensive LLMs. 
Second, it may not always be accurate. LLMs can hallucinate, e.g., the reasoning from one step to another may be flawed. 

An alternative to LLM-as-a-judge is dedicated **NLG (natural language generation) evaluation models** which are generally much smaller than an LLM. 
Finetuned to evaluate specific aspects of LLM generation, these models are much faster, cheaper, and more robust than LLM-as-a-judge. 
For example, Vectara's HHEM is a time-tested model to judge faithfulness. 
It has been downloaded more than 5 million times between August 2024 (when it debut) and August 2025. 

The last, but golden evaluation is **human evaluation**. 
Human evaluators can provide the most accurate and nuanced assessments of LLM responses, taking into account context, intent, and subtlety that automated systems may miss.
However, human evaluation is also the most resource-intensive and time-consuming method. 
It is often used as a final, small-scale check after automated evaluations have been performed.
"""

prompt_template= """
You are a good reader. Answer the query based on the context provided. Give me a short answer. 

Query: {query}
Context: {context}

"""

## Query the LLM

We send the filled prompt to Claude and test how the same context answers different questions, showing the generation step of a RAG pipeline.

In [3]:
model_name = 'claude-sonnet-4-5'
response = anthropic.Anthropic().messages.create(
    model=model_name,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": prompt_template.format(query=query, context=context)}
    ]
)
print (response.content[0].text)

**Human evaluation** is the most accurate way to judge a RAG system when budget and time are not constraints.

According to the context, human evaluators "can provide the most accurate and nuanced assessments of LLM responses, taking into account context, intent, and subtlety that automated systems may miss." It's described as the "golden evaluation" method.


In [4]:
# A different query
query = "What are the common aspects that people judge the RAG system on?"

response = anthropic.Anthropic().messages.create(
    model=model_name,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": prompt_template.format(query=query, context=context)}
    ]
)
print (response.content[0].text)

Based on the context, the two common aspects that people judge RAG systems on are:

1. **Relevance** - does the response answer the query?
2. **Faithfulness** - is the response well supported by the retrieved documents?


In [5]:
# Yet another different query
query = "How many ways to evaluate the RAG system? Just method names."

response = anthropic.Anthropic().messages.create(
    model=model_name,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": prompt_template.format(query=query, context=context)}
    ]
)
print (response.content[0].text)

Based on the context, there are **3 methods** to evaluate the RAG system:

1. **LLM-as-a-judge**
2. **NLG (natural language generation) evaluation models**
3. **Human evaluation**
